## Lab 03 - ANN DL - MLP (Multi-Layered Perceptron) for Classification

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.datasets import make_regression
from sklearn import metrics
from sklearn import decomposition
from sklearn import manifold
from tqdm.notebook import trange, tqdm

#TensorFlow and Keras Libraries
import tensorflow as tf
from tensorflow.keras.layers import Dense, Flatten, Conv2D
from tensorflow.keras import Model
import torch
from torchvision import transforms
from tensorflow.keras.utils import to_categorical
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision
import torchvision.datasets as datasets
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

# Hiding the warnings
import warnings
warnings.filterwarnings('ignore')

# Extra Libraries
import copy
import random
import time

In [2]:
data_path = "DataBase/Class_Data/house-prices.csv"
df = pd.read_csv(data_path)

In [3]:
df.head()

,Home,Price,SqFt,Bedrooms,Bathrooms,Offers,Brick,Neighborhood
0,1,114300,1790,2,2,2,No,East
1,2,114200,2030,4,2,3,No,East
2,3,114800,1740,3,2,1,No,East
3,4,94700,1980,3,2,3,No,East
4,5,119800,2130,3,3,3,No,East


In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Home,128.0,64.500000,37.094474,1.0,32.75,64.5,96.25,128.0
Price,128.0,130427.343750,26868.770371,69100.0,111325.00,125950.0,148250.00,211200.0
SqFt,128.0,2000.937500,211.572431,1450.0,1880.00,2000.0,2140.00,2590.0
Bedrooms,128.0,3.023438,0.725951,2.0,3.00,3.0,3.00,5.0
Bathrooms,128.0,2.445312,0.514492,2.0,2.00,2.0,3.00,4.0
Offers,128.0,2.578125,1.069324,1.0,2.00,3.0,3.00,6.0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128 entries, 0 to 127
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Home          128 non-null    int64 
 1   Price         128 non-null    int64 
 2   SqFt          128 non-null    int64 
 3   Bedrooms      128 non-null    int64 
 4   Bathrooms     128 non-null    int64 
 5   Offers        128 non-null    int64 
 6   Brick         128 non-null    object
 7   Neighborhood  128 non-null    object
dtypes: int64(6), object(2)
memory usage: 8.1+ KB


In [6]:
df.isna().sum()

Home            0
Price           0
SqFt            0
Bedrooms        0
Bathrooms       0
Offers          0
Brick           0
Neighborhood    0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
# Possible target for classification : 
# A) Brick -> Binary Class Classification
# B) Neighborhood -> Multi Class Classification

In [9]:
cat_col = []
for i in df.columns:
    if df[i].dtype == 'O':
        cat_col.append(i)

cat_col

['Brick', 'Neighborhood']

In [10]:
df['Brick'].value_counts()

Brick
No     86
Yes    42
Name: count, dtype: int64

In [11]:
df['Neighborhood'].value_counts()

Neighborhood
East     45
North    44
West     39
Name: count, dtype: int64

In [12]:
df[cat_col].nunique()

Brick           2
Neighborhood    3
dtype: int64

In [13]:
# Now using Label Encoder :
le1 = LabelEncoder()
df['Brick'] = le1.fit_transform(df['Brick'])
df['Brick'].unique()

array([0, 1])

In [14]:
le2 = LabelEncoder()
df['Neighborhood'] = le2.fit_transform(df['Neighborhood'])
df['Neighborhood'].unique()

array([0, 1, 2])

In [15]:
# Dropping column - "Home"
df.drop('Home',axis=1,inplace=True)

In [16]:
# Standardization of data :
sc1 = MinMaxScaler()
df[['Price']] = sc1.fit_transform(df[['Price']])

sc2 = MinMaxScaler()
df[['SqFt']] = sc2.fit_transform(df[['SqFt']])

In [17]:
df['Price'] = round(df['Price'],2)
df['SqFt'] = round(df['SqFt'],2)

In [18]:
df.head()

,Price,SqFt,Bedrooms,Bathrooms,Offers,Brick,Neighborhood
0,0.32,0.30,2,2,2,0,0
1,0.32,0.51,4,2,3,0,0
2,0.32,0.25,3,2,1,0,0
3,0.18,0.46,3,2,3,0,0
4,0.36,0.60,3,3,3,0,0


In [19]:
# Binary Class Classification :
X = df.drop('Brick',axis=1)
y = df['Brick']

In [20]:
# Train-Test Split :
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.33,random_state=42)
print("X_train shape : ",X_train.shape)
print("X_test shape : ",X_test.shape)
print("y_train shape : ",y_train.shape)
print("y_test shape : ",y_test.shape)

X_train shape :  (85, 6)
X_test shape :  (43, 6)
y_train shape :  (85,)
y_test shape :  (43,)


In [21]:
# Building MLP model for Binary Class Classification :
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Dense(45,activation='relu',input_dim = 6))
model.add(tf.keras.layers.Dense(25,activation='relu'))
model.add(tf.keras.layers.Dense(15,activation='relu'))
model.add(tf.keras.layers.Dense(1,activation='sigmoid'))

In [22]:
model.compile(loss=tf.keras.losses.BinaryCrossentropy(), optimizer='adam', metrics=['BinaryAccuracy'])
history = model.fit(X_train, y_train, epochs=500)

Epoch 1/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - BinaryAccuracy: 0.2662 - loss: 0.8247 
Epoch 2/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.2701 - loss: 0.7667 
Epoch 3/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - BinaryAccuracy: 0.2701 - loss: 0.7239
Epoch 4/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - BinaryAccuracy: 0.6086 - loss: 0.6855
Epoch 5/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.7397 - loss: 0.6635 
Epoch 6/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.7299 - loss: 0.6514 
Epoch 7/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - BinaryAccuracy: 0.7143 - loss: 0.6446
Epoch 8/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - BinaryAccuracy: 0.7221 - loss: 0.6360 
Epoch 9/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.6987 - loss: 0.6329 
Epoch 10/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - BinaryAccuracy: 0.6987 - loss: 0.6287 
Epoch 11/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.7065 - loss: 0.6208 
Epoch 1

In [23]:
y_pred = model.predict(X_test)
y_pred.shape

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step


(43, 1)

In [24]:
# y_pred

In [25]:
y_bi_pred = (y_pred > 0.5).astype(int).flatten()
y_bi_pred.shape

(43,)

In [26]:
# Calculating Accuracy, precision, recall, f1-score and Confusion Matrix
accuracy = accuracy_score(y_test, y_bi_pred)
conf_matrix = confusion_matrix(y_test, y_bi_pred)
class_report = classification_report(y_test, y_bi_pred)

print("Accuracy:", accuracy)
print("Confusion Matrix:\n", conf_matrix)
print("Classification Report:\n", class_report)

Accuracy: 0.7906976744186046
Confusion Matrix:
 [[22  3]
 [ 6 12]]
Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.88      0.83        25
           1       0.80      0.67      0.73        18

    accuracy                           0.79        43
   macro avg       0.79      0.77      0.78        43
weighted avg       0.79      0.79      0.79        43



In [27]:
# Multi Class classification :
df.head()

,Price,SqFt,Bedrooms,Bathrooms,Offers,Brick,Neighborhood
0,0.32,0.30,2,2,2,0,0
1,0.32,0.51,4,2,3,0,0
2,0.32,0.25,3,2,1,0,0
3,0.18,0.46,3,2,3,0,0
4,0.36,0.60,3,3,3,0,0


In [28]:
# Train-Test Split :
X = df.drop('Neighborhood',axis=1)
y = df['Neighborhood']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.33,random_state=43)

print("X_train shape : ",X_train.shape)
print("X_test shape : ",X_test.shape)
print("y_train shape : ",y_train.shape)
print("y_test shape : ",y_test.shape)

X_train shape :  (85, 6)
X_test shape :  (43, 6)
y_train shape :  (85,)
y_test shape :  (43,)


In [29]:
y_train.unique()

array([2, 0, 1])

In [30]:
y_test.unique()

array([0, 2, 1])

In [31]:
# Assuming train_y contains class labels like [0, 1, 2, 3, 4]
num_classes = len(set(y_train))  # Automatically detect number of classes

# One-Hot Encode the Labels
train_y = to_categorical(y_train, num_classes)

# Build the MLP Model
model = tf.keras.Sequential()

# Model : 
model.add(tf.keras.layers.Dense(128, activation='relu', input_dim=X_train.shape[1]))
model.add(tf.keras.layers.Dense(64, activation='relu'))
model.add(tf.keras.layers.Dense(32, activation='relu'))
model.add(tf.keras.layers.Dense(num_classes, activation='softmax'))

# Compile the Model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the Model
history = model.fit(X_train, train_y, epochs=500)

# Predict the Output
y_pred = model.predict(X_test)

# Convert Softmax Probabilities to Class Labels
y_pred_classes = y_pred.argmax(axis=1)

Epoch 1/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.2876 - loss: 1.2240 
Epoch 2/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3268 - loss: 1.1108 
Epoch 3/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5401 - loss: 1.0656
Epoch 4/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5499 - loss: 1.0435 
Epoch 5/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5284 - loss: 1.0435
Epoch 6/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5675 - loss: 1.0139
Epoch 7/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6203 - loss: 1.0040 
Epoch 8/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5871 - loss: 0.9988
Epoch 9/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6067 - loss: 0.9827
Epoch 10/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6067 - loss: 0.9541
Epoch 11/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6321 - loss: 0.9297 
Epoch 12/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6555 - l

In [32]:
# Calculating Accuracy, precision, recall, f1-score and Confusion Matrix
accuracy = accuracy_score(y_test, y_pred_classes)
conf_matrix = confusion_matrix(y_test, y_pred_classes)
class_report = classification_report(y_test, y_pred_classes)

print("Accuracy:", accuracy)
print("Confusion Matrix:\n", conf_matrix)
print("Classification Report:\n", class_report)

Accuracy: 0.6511627906976745
Confusion Matrix:
 [[10  9  3]
 [ 3  9  0]
 [ 0  0  9]]
Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.45      0.57        22
           1       0.50      0.75      0.60        12
           2       0.75      1.00      0.86         9

    accuracy                           0.65        43
   macro avg       0.67      0.73      0.68        43
weighted avg       0.69      0.65      0.64        43

